In [1]:
!pip install -U -q \
langchain \
langchain-core \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
google-genai \
faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 863.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sourc

In [14]:
!pip install -q -U google-genai pydantic

In [2]:
import os
import json

from google.colab import userdata

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI,
)

/tmp/ipykernel_771/1769156717.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in Colab Secrets")

print("✅ API Key Loaded")

✅ API Key Loaded


In [4]:
TEXT_FILE = "/content/Narayana_latest.txt"

with open(TEXT_FILE, "r", encoding="utf-8") as file:
    text = file.read()

print("Characters :", len(text))
print(text[:500])

Characters : 26910
PAGE 1
" |IT-JEE/NEET/FOUNDATIONS

YEARS
OF EXCELLENCE

Information Brochure
2020-2022

& Foundation “a Medical AG Engineering

Class VIII, IX, X, NTSE and NEET JEE & Other Engineering
Olympiads Entrance Exams

PAGE 2
Founder’s Message

Education empowers an individual to make
right choices. Right choices in life leads to
success, happiness and satisfaction.

As an educatio


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " "
    ]
)

chunks = text_splitter.split_text(text)

print("Total Chunks :", len(chunks))

Total Chunks : 41


In [6]:
from langchain_core.documents import Document

documents = []

for index, chunk in enumerate(chunks):

    documents.append(
        Document(
            page_content=chunk,
            metadata={
                "chunk_id": index,
                "source": "Narayana.txt"
            }
        )
    )

print(documents[0])

page_content='==============================
PAGE 1
" |IT-JEE/NEET/FOUNDATIONS

YEARS
OF EXCELLENCE

Information Brochure
2020-2022

& Foundation “a Medical AG Engineering

Class VIII, IX, X, NTSE and NEET JEE & Other Engineering
Olympiads Entrance Exams

PAGE 2
Founder’s Message

Education empowers an individual to make
right choices. Right choices in life leads to
success, happiness and satisfaction.

As an educationist, | believe in empowering
students to choose and create a life they
want to live and hence making society better
for all. Right education at the right time can
make student achieve anything they want to.' metadata={'chunk_id': 0, 'source': 'Narayana.txt'}


In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

In [8]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    documents,
    embedding_model
)

vector_db.save_local("faiss_db")

print("✅ FAISS Database Created")

✅ FAISS Database Created


In [9]:
retriever = vector_db.as_retriever(
    search_kwargs={
        "k": 5
    }
)

print("✅ Retriever Ready")

✅ Retriever Ready


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.1
)

print("✅ Gemini Ready")

✅ Gemini Ready


In [29]:
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime


class Lead(BaseModel):

    lead_id: str
    org_id: str

    ai_lead_score: int = Field(
        ge=0,
        le=100
    )

    priority: str
    stage: str

    contact_email: Optional[str] = None
    contact_phone: Optional[str] = None


class ChatbotSession(BaseModel):

    session_id: str
    org_id: str
    lead_id: str

    user_id: Optional[str] = None

    channel: str
    status: str

    started_at: datetime
    ended_at: Optional[datetime] = None


class ChatbotMessage(BaseModel):

    message_id: str
    session_id: str

    sender: str
    content: str

    intent: Optional[str] = None

    confidence_score: Optional[int] = Field(
        default=None,
        ge=0,
        le=100
    )

    model_version: Optional[str] = None

    response_time_ms: Optional[int] = Field(
        default=None,
        ge=0
    )

    created_at: datetime

In [30]:
class AIChatResponse(BaseModel):

    answer: str

    intent: str

    confidence_score: int = Field(
        ge=0,
        le=100
    )

    ai_lead_score: int = Field(
        ge=0,
        le=100
    )

    priority: str

    stage: str

    reason: str

    follow_up_question: str

In [31]:
conversation_history = []

In [32]:
from langchain_core.prompts import ChatPromptTemplate


SYSTEM_PROMPT = """

You are Narayana AI Admission Assistant.

Your job is to answer admission-related questions using ONLY
the retrieved content from Narayana.txt.

==================================================
ANSWER RULES
==================================================

1. Answer ONLY from the Retrieved Context.

2. Do NOT use outside knowledge.

3. Do NOT use internet knowledge.

4. Do NOT guess.

5. Do NOT assume missing information.

6. Do NOT invent:
   - fees
   - courses
   - dates
   - scholarships
   - admission rules
   - addresses
   - phone numbers
   - policies
   - facilities

7. If the required information is not available in the
   Retrieved Context, clearly say:

   "I couldn't find this information in the uploaded brochure."

8. Keep the answer concise and professional.

9. Prefer 30-60 words.

10. Maximum answer length is 80 words.

11. Do not unnecessarily repeat information.

12. Do not answer unrelated questions using outside knowledge.

==================================================
HALLUCINATION CONTROL
==================================================

Before answering, verify that the answer is supported
by the Retrieved Context.

If the Retrieved Context does not contain sufficient
information:

DO NOT GUESS.

Return:

"I couldn't find this information in the uploaded brochure."

==================================================
CONVERSATION HISTORY
==================================================

Use the complete conversation history to understand
the user's admission journey.

Do not evaluate the current question in isolation.

==================================================
AI LEAD SCORE
==================================================

Generate an AI Lead Score between 0 and 100.

0-20:
Greeting, casual browsing, no clear admission intent.

21-40:
General institute or course information.

41-60:
Eligibility, admission process, syllabus,
course duration, general planning.

61-80:
Fees, scholarship, branch, hostel,
course comparison, detailed admission questions.

81-100:
Registration, seat availability, payment,
counsellor request, callback request,
or clear intention to join.

IMPORTANT:

The Lead Score must consider the COMPLETE
conversation history.

Example progression:

Course information
→ Fee
→ Scholarship
→ Admission
→ Registration

should normally increase the Lead Score.

Do not reset the score for every question.

==================================================
PRIORITY
==================================================

Use:

Low
Medium
High
Critical

==================================================
STAGE
==================================================

Use:

New
Interested
Qualified
Ready_to_Join
Converted

==================================================
FOLLOW-UP
==================================================

Generate ONE relevant follow-up question.

The question should help understand the
user's admission requirement.

==================================================
OUTPUT
==================================================

Return the response using the provided
Pydantic structured output schema.

"""


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),

    ("human", """

Conversation History:

{history}

--------------------------------

Retrieved Context:

{context}

--------------------------------

Current User Question:

{question}

""")
])

In [33]:
structured_llm = llm.with_structured_output(
    AIChatResponse
)

In [34]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GOOGLE_API_KEY
)

In [35]:
def format_docs(docs):

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [36]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {
        "context": retriever | format_docs,

        "history": lambda _: conversation_history,

        "question": RunnablePassthrough()
    }

    | prompt

    | structured_llm
)

In [37]:
response = rag_chain.invoke(
    "Tell me about IIT JEE course"
)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


answer='Narayana offers nDigital IIT-JEE programs to prepare students for JEE Mains and Advanced. Options include the 2-year nElite Program (CRM-O1) for Class XI and 1-year nElite Programs (CRM-O2) for Class XII and XII+ students. Courses provide study material in Physics, Chemistry, and Math, regular tests, and the All India Narayana Test Series (AINTS).' intent='Course Information' confidence_score=95 ai_lead_score=30 priority='Low' stage='Interested' reason='User is asking for general information about the IIT JEE course options.' follow_up_question='Which class are you or your child currently studying in?'


In [38]:
json_data = response.model_dump()
print(json_data)

{'answer': 'Narayana offers nDigital IIT-JEE programs to prepare students for JEE Mains and Advanced. Options include the 2-year nElite Program (CRM-O1) for Class XI and 1-year nElite Programs (CRM-O2) for Class XII and XII+ students. Courses provide study material in Physics, Chemistry, and Math, regular tests, and the All India Narayana Test Series (AINTS).', 'intent': 'Course Information', 'confidence_score': 95, 'ai_lead_score': 30, 'priority': 'Low', 'stage': 'Interested', 'reason': 'User is asking for general information about the IIT JEE course options.', 'follow_up_question': 'Which class are you or your child currently studying in?'}


In [39]:
from IPython.display import JSON, display

display(
    JSON(
        response.model_dump(),
        expanded=True
    )
)

<IPython.core.display.JSON object>

In [40]:
from datetime import datetime
import uuid

In [41]:
session_id = "SESSION_001"
lead_id = "LEAD_001"
org_id = "ORG_001"
user_id = "USER_001"

In [42]:
lead = Lead(

    lead_id=lead_id,

    org_id=org_id,

    ai_lead_score=response.ai_lead_score,

    priority=response.priority,

    stage=response.stage,

    contact_email=None,

    contact_phone=None
)

In [43]:
session = ChatbotSession(

    session_id=session_id,

    org_id=org_id,

    lead_id=lead_id,

    user_id=user_id,

    channel="Web",

    status="Active",

    started_at=datetime.now(),

    ended_at=None
)

In [44]:
user_message = ChatbotMessage(

    message_id=str(uuid.uuid4()),

    session_id=session_id,

    sender="user",

    content="Tell me about IIT JEE course",

    intent=response.intent,

    confidence_score=None,

    model_version="gemini-3.6-flash",

    response_time_ms=None,

    created_at=datetime.now()
)

In [45]:
assistant_message = ChatbotMessage(

    message_id=str(uuid.uuid4()),

    session_id=session_id,

    sender="assistant",

    content=response.answer,

    intent=response.intent,

    confidence_score=response.confidence_score,

    model_version="gemini-3.6-flash",

    response_time_ms=None,

    created_at=datetime.now()
)

In [46]:
database_json = {

    "leads": lead.model_dump(mode="json"),

    "chatbot_sessions": session.model_dump(mode="json"),

    "chatbot_messages": [

        user_message.model_dump(mode="json"),

        assistant_message.model_dump(mode="json")

    ]
}

In [47]:
display(
    JSON(
        database_json,
        expanded=True
    )
)

<IPython.core.display.JSON object>

In [24]:
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime

In [25]:
class Lead(BaseModel):

    lead_id: str = Field(
        description="Unique lead identifier"
    )

    org_id: str = Field(
        description="Organization identifier"
    )

    ai_lead_score: int = Field(
        ge=0,
        le=100,
        description="AI-generated lead score based on complete conversation history"
    )

    priority: str = Field(
        description="Lead priority such as Low, Medium, High, Critical"
    )

    stage: str = Field(
        description="Current lead stage such as New, Interested, Qualified, Converted"
    )

    contact_email: Optional[str] = Field(
        default=None,
        description="Lead email address"
    )

    contact_phone: Optional[str] = Field(
        default=None,
        description="Lead phone number"
    )

In [26]:
class ChatbotSession(BaseModel):

    session_id: str = Field(
        description="Unique chatbot session identifier"
    )

    org_id: str = Field(
        description="Organization identifier"
    )

    lead_id: str = Field(
        description="Associated lead identifier"
    )

    user_id: Optional[str] = Field(
        default=None,
        description="User identifier"
    )

    channel: str = Field(
        description="Chat channel such as Web, Mobile, WhatsApp, etc."
    )

    status: str = Field(
        description="Session status such as Active, Completed, Abandoned"
    )

    started_at: datetime = Field(
        description="Session start timestamp"
    )

    ended_at: Optional[datetime] = Field(
        default=None,
        description="Session end timestamp"
    )

In [27]:
class ChatbotMessage(BaseModel):

    message_id: str = Field(
        description="Unique message identifier"
    )

    session_id: str = Field(
        description="Chatbot session identifier"
    )

    sender: str = Field(
        description="Message sender: user or assistant"
    )

    content: str = Field(
        description="User question or chatbot answer"
    )

    intent: Optional[str] = Field(
        default=None,
        description="Detected user intent"
    )

    confidence_score: Optional[int] = Field(
        default=None,
        ge=0,
        le=100,
        description="AI confidence score"
    )

    model_version: Optional[str] = Field(
        default=None,
        description="LLM model version used to generate the response"
    )

    response_time_ms: Optional[int] = Field(
        default=None,
        ge=0,
        description="LLM response time in milliseconds"
    )

    created_at: datetime = Field(
        description="Message creation timestamp"
    )

In [28]:
from langchain_core.prompts import ChatPromptTemplate


SYSTEM_PROMPT = """
You are Narayana AI Admission Assistant.

Your job is to answer the user's admission-related questions using ONLY
the retrieved content from Narayana.txt.

=============================
ANSWER RULES
=============================

1. Use ONLY the Retrieved Context for factual answers.

2. Do NOT use outside knowledge.

3. Do NOT use internet knowledge.

4. Do NOT guess.

5. Do NOT assume missing information.

6. Do NOT invent fees, dates, courses, scholarships,
   admission rules, addresses, phone numbers, facilities,
   or policies.

7. If the answer is not clearly available in the
   Retrieved Context, say:

   "I couldn't find this information in the uploaded brochure."

8. Keep the answer concise and professional.

9. Prefer 30-60 words.

10. Maximum answer length is 80 words.

11. Do not repeat information unnecessarily.

12. Do not provide long explanations unless the user asks
    for detailed information.

13. Do not answer unrelated questions using general knowledge.

=============================
CONTEXT GROUNDING
=============================

Before answering, verify that the answer is supported
by the Retrieved Context.

If the Retrieved Context does not contain sufficient
information, do not guess.

=============================
CONVERSATION HISTORY
=============================

Use the complete Conversation History to understand
the user's journey.

Do not treat the current question as an isolated question.

=============================
LEAD SCORE
=============================

Generate a Lead Score from 0 to 100 based primarily on
the user's admission intent and the complete conversation.

0-20:
Greeting, casual browsing, no clear admission intent.

21-40:
General institute or course information.

41-60:
Eligibility, admission process, syllabus,
course duration, general planning.

61-80:
Fees, scholarship, branch, hostel,
course comparison, detailed admission questions.

81-100:
Registration, seat availability, payment,
counsellor request, callback request,
or clear intention to join.

IMPORTANT:

The score must consider the COMPLETE conversation.

If the user progresses from:

course information
→ fee
→ scholarship
→ admission
→ registration

the lead score should generally increase.

Do not reset the score for every question.

=============================
LEAD GRADE
=============================

0-20   = Cold
21-40  = Curious
41-60  = Interested
61-80  = Hot
81-100 = Ready for Admission

=============================
LEAD STATUS
=============================

Use an appropriate status such as:

Browsing
Interested
Qualified
High Probability
Ready To Join

=============================
HALLUCINATION CONTROL
=============================

If information is missing from the Retrieved Context:

DO NOT COMPLETE THE ANSWER FROM YOUR OWN KNOWLEDGE.

Instead clearly state that the information
was not found in the uploaded brochure.

=============================
FOLLOW-UP QUESTION
=============================

Ask only ONE relevant follow-up question.

The follow-up question should help understand
the user's admission requirement or move the
conversation toward admission.

Do not ask unnecessary questions.

=============================
COUNSELLOR ACTION
=============================

next_best_action should be practical and relevant
to the current lead stage.

Examples:

"Provide course details"

"Explain fee details"

"Explain scholarship information"

"Ask about preferred course"

"Offer admission registration guidance"

"Recommend counsellor follow-up"

=============================
OUTPUT
=============================

Return the response according to the provided
Pydantic structured output schema.
"""


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),

    ("human", """
Conversation History:

{history}

--------------------------------

Retrieved Context:

{context}

--------------------------------

Current User Question:

{question}
""")
])

In [17]:
structured_llm = llm.with_structured_output(
    AdmissionResponse
)

In [18]:
conversation_history = []

In [19]:
def format_docs(docs):

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [20]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {
        "context": retriever | format_docs,

        "history": lambda _: conversation_history,

        "question": RunnablePassthrough()
    }

    | prompt

    | structured_llm
)

In [21]:
from datetime import datetime


SESSION_ID = "SESSION_001"


def chatbot(question):

    response = rag_chain.invoke(question)

    result = response.model_dump()

    result["session_id"] = SESSION_ID

    result["timestamp"] = datetime.now().isoformat()

    conversation_history.append({

        "question": result["question"],

        "answer": result["chat"]["answer"],

        "intent": result["lead"]["intent"],

        "category": result["lead"]["category"],

        "lead_score": result["lead"]["lead_score"],

        "lead_grade": result["lead"]["lead_grade"],

        "lead_status": result["lead"]["lead_status"]

    })

    return result

In [22]:
response = chatbot(
    "Tell me about IIT JEE course"
)

response

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'session_id': 'SESSION_001',
 'question': 'Tell me about IIT JEE course',
 'chat': {'answer': "Narayana's nDigital IIT-JEE program prepares students for JEE Main, Advanced, and board exams. It offers courses for Class 11, 12, and 12+ students with study materials for Physics, Chemistry, and Math, regular test analysis, and All India Narayana Test Series (AINTS). Admission is available based on previous marks or NACST.",
  'follow_up_question': 'Which class are you currently studying in?'},
 'lead': {'intent': 'Inquiring about IIT JEE course details',
  'category': 'Course',
  'lead_score': 30,
  'lead_grade': 'Curious',
  'lead_status': 'Interested',
  'confidence': 95,
  'student_interest': 'IIT-JEE',
  'admission_probability': 35,
  'reason': 'User is asking general course information regarding IIT-JEE programs.',
  'next_best_action': 'Ask about preferred class or program to guide admission process'},
 'retrieval': {'source': 'Narayana.txt',
  'page_number': 8,
  'retrieved_chunks'

In [23]:
from IPython.display import JSON, display

display(
    JSON(
        response,
        expanded=True
    )
)

<IPython.core.display.JSON object>

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI Admission Assistant.

Answer ONLY from the provided context.

If the answer is not available, say:
"I couldn't find this information."

Return ONLY valid JSON.

{{
    "question":"",
    "answer":"",
    "confidence":"",
    "source":"Narayana.txt"
}}

Context:
{context}

Question:
{question}
""")

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [13]:
question = "What is the admission process?"

response = rag_chain.invoke(question)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```json
{
    "question": "What is the admission process?",
    "answer": "The admission process may vary according to the course. Students can get direct admission based on the percentage of marks/grades secured in the last qualifying/previous class or by competing in scholarship/reward exams such as N-ACST (Narayana Admission Cum Scholarship Test).",
    "confidence": "High",
    "source": "Narayana.txt"
}
```


In [ ]:
question = "What documents are required for admission?"

response = rag_chain.invoke(question)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
    "question": "What documents are required for admission?",
    "answer": "The required documents for admission include: 1. DD/PDCS of course fee in favour of NARAYANA Institute, 2. Two recent passport size color photographs (plus 3 extra passport size color photographs stapled with the form), 3. Self-attested photocopy of mark sheet (Class VII / VIII / IX / X / XII), and 4. Proof of Scholarship.",
    "confidence": "High",
    "source": "Narayana.txt"
}


In [ ]:
question = "What is the Founder Name?"

response = rag_chain.invoke(question)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
    "question": "What is the Founder Name?",
    "answer": "Dr. P. Narayana",
    "confidence": "High",
    "source": "Narayana.txt"
}


In [ ]:
question = "How many Course Available and what are there fee details?"

response = rag_chain.invoke(question)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```json
{
    "question": "How many Course Available and what are there fee details?",
    "answer": "The context does not state the total count of available courses directly, but it lists 9 courses along with their fee details:\n\n1. Foundation 8 Narayana nDigital (Foundation Courses - 8th): Fee 39,999\n2. Foundation 9 Narayana nDigital (Foundation Courses - 9th): Fee 39,999\n3. Foundation 10 1 Narayana nDigital (Foundation Courses - 10th): Fee 44,999\n4. JEE (Main & Adv.) Two Year Courses (XI & XII): Fee 1,59,999\n5. NEET Two Year Courses (XI & XII): Fee 1,59,999\n6. JEE (Main & Adv.) Courses (XII): Fee 82,999\n7. NEET Courses (XII): Fee 82,999\n8. JEE (Main & Adv.) 12+ One Year Repeater Courses: Fee 95,999\n9. NEET 12+ One Year Repeater Courses: Fee 99,999",
    "confidence": "95%",
    "source": "Narayana.txt"
}
```


In [ ]:
question = "what is the criteria for NEET?"

response = rag_chain.invoke(question)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```json
{
    "question": "what is the criteria for NEET?",
    "answer": "For the Two-year integrated course for NEET/JEE, students must have 70% and above aggregate marks or B1 and above grade in Science and Maths in class 10th board. For the One-year integrated course for NEET/JEE, students must have 70% and above aggregate marks or B1 and above grade in Science and Maths in class 10, or an aggregate of 65% in PCB/PCM in class 11th. Direct admission is granted based on the marks/grade obtained in the previous year's class or through N-ACST.",
    "confidence": "High",
    "source": "Narayana.txt"
}
```


In [ ]:
question = "What are the payment methods?"

response = rag_chain.invoke(question)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
    "question": "What are the payment methods?",
    "answer": "The payment methods are Paytm, UPI, NETBANKING, and CREDIT CARDS.",
    "confidence": "100%",
    "source": "Narayana.txt"
}


In [ ]:
question = "What scholarships are available?"

response = rag_chain.invoke(question)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
    "question": "What scholarships are available?",
    "answer": "Narayana offers scholarship tests which include the Reward exam and N-ACST (Narayana Admission Cum Scholarship Test). The scholarship can be up to 90%.",
    "confidence": "High",
    "source": "Narayana.txt"
}


In [ ]:
PROMPT = """
You are an Admission Counsellor AI.

Answer ONLY from the retrieved context.

Also evaluate the user's buying intent.

Generate ONLY valid JSON.

{
    "session_id":"",
    "question":"",
    "answer":"",
    "intent":"",
    "category":"",
    "lead_score":0,
    "lead_grade":"",
    "lead_status":"",
    "confidence":0,
    "source":"",
    "page":"",
    "retrieved_chunks":[],
    "next_best_action":"",
    "follow_up_question":"",
    "timestamp":""
}

Lead Score Rules

0-20 = Cold
21-50 = Warm
51-80 = Hot
81-100 = Highly Interested

Context

{context}

Question

{question}
"""

In [ ]:
conversation_history = []

conversation_history.append({

    "question": question,

    "answer": answer,

    "lead_score": result["lead_score"],

    "intent": result["intent"]

})

NameError: name 'answer' is not defined

In [ ]:
conversation_history = []

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime


class ChatResponse(BaseModel):

    answer: str = Field(
        description="Answer returned to the user"
    )

    follow_up_question: str = Field(
        description="Next question to continue the conversation"
    )


class LeadProfile(BaseModel):

    intent: str = Field(
        description="Detected user intent"
    )

    category: str = Field(
        description="Course Inquiry, Admission, Fee, Scholarship, Registration, Hostel, etc."
    )

    lead_score: int = Field(
        ge=0,
        le=100,
        description="Lead score between 0 and 100"
    )

    lead_grade: str = Field(
        description="Cold, Curious, Interested, Hot, Ready for Admission"
    )

    lead_status: str = Field(
        description="Browsing, Interested, Qualified, Ready to Join"
    )

    confidence: int = Field(
        ge=0,
        le=100,
        description="Model confidence percentage"
    )

    student_interest: str = Field(
        description="Detected course interest"
    )

    admission_probability: int = Field(
        ge=0,
        le=100,
        description="Probability that the student will take admission"
    )

    reason: str = Field(
        description="Reason for assigning the lead score"
    )

    next_best_action: str = Field(
        description="Recommended counsellor action"
    )


class RetrievalMetadata(BaseModel):

    source: str = Field(
        description="Document name"
    )

    page_number: int = Field(
        description="Source page number"
    )

    retrieved_chunks: List[int] = Field(
        description="Retrieved chunk IDs"
    )


class AdmissionResponse(BaseModel):

    session_id: str = Field(
        description="Unique conversation session"
    )

    timestamp: str = Field(
        default_factory=lambda: datetime.now().isoformat()
    )

    question: str = Field(
        description="Current user question"
    )

    chat: ChatResponse

    lead: LeadProfile

    retrieval: RetrievalMetadata

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
[
    (
        "system",
        """
You are Narayana AI Admission Assistant.

Your primary goal is to answer admission-related questions using ONLY the retrieved context from the uploaded Narayana brochure.

===========================
Response Rules
===========================

1. Answer ONLY from the retrieved context.

2. Never use your own knowledge.

3. Never assume or guess information.

4. Never generate information that is not present in the retrieved context.

5. If the answer is not available in the retrieved context, reply exactly:

"I couldn't find this information in the uploaded brochure."

6. Do NOT mention information from the internet.

7. Do NOT mention general educational knowledge.

8. Do NOT fabricate fees, dates, courses, addresses, phone numbers or policies.

9. Do NOT answer unrelated questions.

===========================
Response Style
===========================

• Be professional.

• Be concise.

• Use simple English.

• Answer in 2-5 sentences.

• Avoid long explanations.

• Avoid repeating information.

• Avoid unnecessary greetings.

• Give direct answers.

===========================
Context Rules
===========================

Use ONLY

- Conversation History
- Retrieved Context

If there is a conflict,

always trust the Retrieved Context.

===========================
Hallucination Prevention
===========================

If confidence is low,

or the information is incomplete,

say

"I couldn't find this information in the uploaded brochure."

Never guess.

===========================
Lead Analysis
===========================

Use the complete conversation history to determine

- Intent
- Category
- Lead Score
- Lead Grade
- Lead Status
- Student Interest
- Admission Probability

Do not use retrieved context for Lead Score.

Lead Score should be based on user behaviour and conversation progression.

===========================
Answer Length
===========================

Maximum 40 words.

Prefer 7-20 words.

===========================
Output
===========================

Return the response using the provided structured output schema.
"""
    ),

    (
        "human",
        """
Conversation History

{history}

-------------------------------------

Retrieved Context

{context}

-------------------------------------

Current Question

{question}
"""
    )
]
)

In [ ]:
structured_llm = llm.with_structured_output(AdmissionResponse)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "history": lambda _: conversation_history,
        "question": RunnablePassthrough()
    }
    | prompt
    | structured_llm
)

In [ ]:
from IPython.display import display

conversation_history = []

def chatbot(question):

    response = rag_chain.invoke(question)

    response.session_id = "SESSION_001"

    conversation_history.append(
        response.model_dump()
    )

    return response

In [ ]:
chatbot("Tell me about IIT JEE Course")

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AdmissionResponse(session_id='SESSION_001', timestamp='2023-10-25T10:00:00Z', question='Tell me about IIT JEE Course', chat=ChatResponse(answer='Narayana offers nDigital IIT-JEE courses to prepare students for JEE (Mains & Advance) and other engineering entrance exams alongside school/board exams. Key programs include: 1) CRM-O1: Two-year nElite Program for Class 11 studying students. 2) CRM-O2: One-year nElite Program for Class 12 studying students. 3) One-year nElite Program for Class 12 passed (XII+) students. Course highlights include comprehensive study material for Physics, Chemistry, and Mathematics, regular tests with SMS result analysis sent to parents, All India Narayana Test Series (AINTS), model test papers, and online test series. Admission is granted directly based on previous class marks/grades or through NACST.', follow_up_question='Which class are you currently studying in so I can recommend the right program for you?'), lead=LeadProfile(intent='Course Inquiry', catego

AdmissionResponse(session_id='SESSION_001', timestamp='2023-10-25T10:00:00Z', question='Tell me about IIT JEE Course', chat=ChatResponse(answer='Narayana offers nDigital IIT-JEE courses to prepare students for JEE (Mains & Advance) and other engineering entrance exams alongside school/board exams. Key programs include: 1) CRM-O1: Two-year nElite Program for Class 11 studying students. 2) CRM-O2: One-year nElite Program for Class 12 studying students. 3) One-year nElite Program for Class 12 passed (XII+) students. Course highlights include comprehensive study material for Physics, Chemistry, and Mathematics, regular tests with SMS result analysis sent to parents, All India Narayana Test Series (AINTS), model test papers, and online test series. Admission is granted directly based on previous class marks/grades or through NACST.', follow_up_question='Which class are you currently studying in so I can recommend the right program for you?'), lead=LeadProfile(intent='Course Inquiry', catego

In [ ]:
chatbot("Hi")

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AdmissionResponse(session_id='SESSION_001', timestamp='2023-10-25T10:00:00Z', question='Hi', chat=ChatResponse(answer='Hello! Welcome to Narayana AI Admission Assistant. How can I assist you today with course admissions, JEE, NEET, or Foundation programs?', follow_up_question='Are you looking for admission in JEE, NEET, or Foundation courses?'), lead=LeadProfile(intent='Greeting', category='General Inquiry', lead_score=10, lead_grade='Cold', lead_status='Browsing', confidence=100, student_interest='Not Specified', admission_probability=10, reason='User initiated conversation with a simple greeting.', next_best_action='Ask user about their target course or class.'), retrieval=RetrievalMetadata(source='Information Brochure 2020-2022', page_number=1, retrieved_chunks=[1]))

In [ ]:
response = chatbot("Hi")

lead_json = response.model_dump()

print(lead_json)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'session_id': 'SESSION_001', 'timestamp': '2023-10-25T10:00:00Z', 'question': 'Hi', 'chat': {'answer': 'Hello! How can I help you with Narayana admissions today?', 'follow_up_question': 'Are you looking for information regarding IIT-JEE, NEET, or Foundation programs?'}, 'lead': {'intent': 'Greeting', 'category': 'General Inquiry', 'lead_score': 10, 'lead_grade': 'Cold', 'lead_status': 'Browsing', 'confidence': 95, 'student_interest': 'General', 'admission_probability': 10, 'reason': 'User initiated conversation with a basic greeting.', 'next_best_action': "Ask for student's target course or current class."}, 'retrieval': {'source': 'Narayana Information Brochure 2020-2022', 'page_number': 1, 'retrieved_chunks': [1]}}


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
[
(
"system",
"""
You are an AI Admission Counsellor.

Answer ONLY from the retrieved context.

Analyze the conversation history.

Generate:

- answer
- intent
- lead_score
- lead_grade
- lead_status
- confidence
- reason
- next_best_action
- follow_up_question

Lead Score

0-20 Cold

21-40 Curious

41-60 Interested

61-80 Hot

81-100 Ready for Admission

IMPORTANT

Return ONLY valid JSON.

DO NOT wrap JSON in markdown.

DO NOT explain your answer.

"""
),

(
"human",
"""
Conversation History

{history}

Retrieved Context

{context}

Question

{question}
"""
)

]
)

In [ ]:
print(prompt.input_variables)

['context', 'history', 'question']


In [ ]:
def format_docs(docs):

    return "\n\n".join(

        doc.page_content

        for doc in docs

    )

In [ ]:
import json

from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import RunnablePassthrough

rag_chain = (

    {

        "context": retriever | format_docs,

        "question": RunnablePassthrough(),

        "history": lambda x: json.dumps(
            conversation_history,
            indent=2
        )

    }

    | prompt

    | llm

    | StrOutputParser()

)

In [ ]:
import json

from datetime import datetime

from IPython.display import JSON, display

In [ ]:
def chatbot(question):

    response = rag_chain.invoke(question)

    print("\nRaw Response\n")
    print(response)

    response = response.strip()

    if response.startswith("```json"):

        response = response.replace(
            "```json",
            ""
        ).replace(
            "```",
            ""
        ).strip()

    elif response.startswith("```"):

        response = response.replace(
            "```",
            ""
        ).strip()

    result = json.loads(response)

    result["timestamp"] = datetime.now().isoformat()

    conversation_history.append({

        "question": result.get("question"),

        "answer": result.get("answer"),

        "intent": result.get("intent"),

        "lead_score": result.get("lead_score"),

        "lead_grade": result.get("lead_grade"),

        "timestamp": result["timestamp"]

    })

    display(JSON(result))

    return result

In [ ]:
chatbot("Tell me about IIT JEE course")

KeyError: 'Input to ChatPromptTemplate is missing variables {\'\\n    "question"\'}.  Expected: [\'\\n    "question"\', \'context\', \'history\', \'question\'] Received: [\'context\', \'question\', \'history\']\nNote: if you intended {\n    "question"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{\n    "question"}}\'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT '